# 02 — Stratified Split Baseline: Decision Tree (No Angkatan)

Fase 4 CRISP-DM | Variasi: Stratified random split (80/20) menggantikan temporal split.

**Tujuan:** Menguji apakah menambah sampel negatif di training (14 → ~54) via stratified split memperbaiki recall kelas 0.

**Perbandingan dengan Notebook 01:**
| | 01 (Temporal) | 02 (Stratified) |
|---|---|---|
| Split logic | angkatan ≤ 2021 / > 2021 | random 80/20, stratify=target |
| Train negatif | 14 (3.7%) | ~54 (11.2%) |
| Test negatif | 54 (23.4%) | ~14 (11.2%) |
| Validitas | Simulasi prediksi masa depan | Generalisasi statistik |

**Catatan:** Fitur `angkatan` dan `program` di-drop untuk mencegah data leakage (angkatan 2023 = semua target=0).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate, learning_curve
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_auc_score, ConfusionMatrixDisplay
)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 120
plt.rcParams['savefig.bbox'] = 'tight'

print("Library loaded.")
print(f"  pandas : {pd.__version__}")
print(f"  sklearn: imported")

In [ ]:
# Load clean dataset
df = pd.read_csv('../3-data-preparation/dataset_clean.csv')
print(f"Shape  : {df.shape}")
print(f"Target : {dict(df['target'].value_counts())}")
print(f"Rate   : {df['target'].mean()*100:.2f}% on-time")

# Stratified split: 80% train, 20% test
X = df.drop(columns=['target'])
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n{'='*50}")
print(f"STRATIFIED SPLIT (80/20, random_state=42)")
print(f"{'='*50}")
print(f"Train: {X_train.shape[0]} rows  |  Target: {dict(y_train.value_counts())}  "
      f"|  {y_train.value_counts(normalize=True)[0]*100:.1f}% neg")
print(f"Test:  {X_test.shape[0]} rows  |  Target: {dict(y_test.value_counts())}  "
      f"|  {y_test.value_counts(normalize=True)[0]*100:.1f}% neg")

# Drop angkatan (data leakage risk: 2023 = all target=0)
# Also drop program (zero importance in preliminary runs)
drop_cols = ['angkatan', 'program']
print(f"\nDropping features: {drop_cols}")
X_train = X_train.drop(columns=drop_cols)
X_test  = X_test.drop(columns=drop_cols)
print(f"Remaining features: {list(X_train.columns)}")
print(f"Feature count: {len(X_train.columns)}")

In [ ]:
# Baseline: DecisionTreeClassifier default
dt_strat = DecisionTreeClassifier(random_state=42)
dt_strat.fit(X_train, y_train)

print("Baseline DT trained (stratified split).")
print(f"Tree depth   : {dt_strat.get_depth()}")
print(f"Tree leaves  : {dt_strat.get_n_leaves()}")
print(f"Node count   : {dt_strat.tree_.node_count}")
print(f"Features used: {(dt_strat.feature_importances_ > 0).sum()} / {len(X_train.columns)}")

In [ ]:
# Predict
y_pred_train = dt_strat.predict(X_train)
y_pred_test  = dt_strat.predict(X_test)

print("=" * 55)
print("CLASSIFICATION REPORT — TRAIN")
print("=" * 55)
print(classification_report(y_train, y_pred_train, target_names=['Tidak Tepat', 'Tepat Waktu']))

print("=" * 55)
print("CLASSIFICATION REPORT — TEST")
print("=" * 55)
print(classification_report(y_test, y_pred_test, target_names=['Tidak Tepat', 'Tepat Waktu']))

# Key metrics
print("\nKEY METRICS (Kelas 0 = Tidak Tepat)")
print('-' * 55)
for name, y_true, y_pred in [('Train', y_train, y_pred_train), ('Test', y_test, y_pred_test)]:
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, pos_label=0, zero_division=0)
    rec  = recall_score(y_true, y_pred, pos_label=0)
    f1   = f1_score(y_true, y_pred, pos_label=0)
    auc  = roc_auc_score(y_true, y_pred)
    print(f"{name:<8} | Acc={acc:.4f} | Prec(0)={prec:.4f} | Recall(0)={rec:.4f} | "
          f"F1(0)={f1:.4f} | AUC={auc:.4f}")

In [ ]:
# Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (name, y_true, y_pred) in zip(
    axes,
    [('Training Set', y_train, y_pred_train), ('Test Set', y_test, y_pred_test)]
):
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Tidak Tepat', 'Tepat Waktu'])
    disp.plot(ax=ax, cmap='Blues', colorbar=False, values_format='d')
    ax.set_title(f'Confusion Matrix — {name}')

plt.tight_layout()
plt.show()

In [ ]:
# Stratified 10-fold CV
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
cv_results = cross_validate(
    dt_strat, X_train, y_train, cv=cv,
    scoring=['accuracy', 'precision', 'recall', 'f1', 'roc_auc'],
    return_train_score=True
)

print("=" * 55)
print("10-FOLD CROSS-VALIDATION (Train only)")
print("=" * 55)
for metric in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']:
    test_scores = cv_results[f'test_{metric}']
    train_scores = cv_results[f'train_{metric}']
    print(f"\n{metric.upper():<14}  Train: {train_scores.mean():.4f} +- {train_scores.std():.4f}")
    print(f"{'':<14}  Test:  {test_scores.mean():.4f} +- {test_scores.std():.4f}")

print(f"\n{'─'*50}")
print(f"{'Metric':<14}  {'Train':<12} {'Test':<12} {'Gap':<12}")
print(f"{'─'*50}")
for metric in ['accuracy', 'recall', 'f1', 'roc_auc']:
    gap = cv_results[f'train_{metric}'].mean() - cv_results[f'test_{metric}'].mean()
    print(f"{metric:<14}  {cv_results[f'train_{metric}'].mean():<12.4f} "
          f"{cv_results[f'test_{metric}'].mean():<12.4f} {gap:<+12.4f}")

In [ ]:
# Feature importance
importances = dt_strat.feature_importances_
feat_imp = pd.DataFrame({
    'feature': X_train.columns,
    'importance': importances
}).sort_values('importance', ascending=False)

print("FEATURE IMPORTANCE (Stratified DT Baseline)")
print("=" * 45)
for i, row in feat_imp.iterrows():
    bar = '█' * int(row['importance'] * 50)
    print(f"  {row['feature']:<25} {row['importance']:.4f}  {bar}")

# Plot
fig, ax = plt.subplots(figsize=(10, 8))
colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(feat_imp)))
feat_plot = feat_imp.iloc[::-1]
bars = ax.barh(feat_plot['feature'], feat_plot['importance'], color=colors[::-1])
for bar, val in zip(bars, feat_plot['importance']):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)
ax.set_xlabel('Importance', fontsize=12)
ax.set_title('Feature Importance — Stratified DT Baseline', fontsize=14, fontweight='bold')
ax.set_xlim(0, feat_imp['importance'].max() * 1.15)
sns.despine(left=True)
plt.tight_layout()
plt.show()

# Zero importance
zero_imp = feat_imp[feat_imp['importance'] == 0]
if len(zero_imp) > 0:
    print(f"\nFeatures with ZERO importance ({len(zero_imp)}):")
    for _, row in zero_imp.iterrows():
        print(f"  - {row['feature']}")
else:
    print("\nAll features have non-zero importance.")

In [ ]:
# Visualize tree (max_depth=4)
fig, ax = plt.subplots(figsize=(20, 10))
plot_tree(
    dt_strat, feature_names=X_train.columns,
    class_names=['Tidak Tepat', 'Tepat Waktu'],
    filled=True, rounded=True, fontsize=8, max_depth=4,
    impurity=False, proportion=True, ax=ax
)
ax.set_title('Decision Tree — Stratified Split (max_depth=4 shown)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()
print(f"Full tree depth: {dt_strat.get_depth()}")

In [ ]:
tree_rules = export_text(dt_strat, feature_names=list(X_train.columns))
lines = tree_rules.split('\n')
print(f"Total rules/lines: {len(lines)}")
print(f"\nFirst 80 lines:\n{'='*60}")
print('\n'.join(lines[:80]))
if len(lines) > 80:
    print(f"\n... ({len(lines) - 80} more lines)")

with open('rules_stratified.txt', 'w') as f:
    f.write(tree_rules)

In [ ]:
print("\n" + "=" * 70)
print("COMPARISON: Temporal (01) vs Stratified (02) Baseline")
print("=" * 70)
print(f"{'Metrik':<18} {'Temporal (01)':<18} {'Stratified (02)':<18} {'Delta':<12}")
print(f"{'─'*70}")

# Values from 01-baseline
temporal = {'accuracy': 0.775, 'recall(0)': 0.037, 'f1(0)': 0.071, 'auc': 0.519}
stratified = {
    'accuracy': accuracy_score(y_test, y_pred_test),
    'recall(0)': recall_score(y_test, y_pred_test, pos_label=0),
    'f1(0)': f1_score(y_test, y_pred_test, pos_label=0),
    'auc': roc_auc_score(y_test, y_pred_test),
}

for m in ['accuracy', 'recall(0)', 'f1(0)', 'auc']:
    delta = stratified[m] - temporal[m]
    symbol = '↑' if delta > 0 else '↓'
    print(f"{m:<18} {temporal[m]:<18.4f} {stratified[m]:<18.4f} {symbol} {abs(delta):.4f}")

print(f"{'─'*70}")
print(f"\nModel complexity comparison:")
print(f"  Temporal (14 neg)  : depth=9, leaves=21, nodes=41")
print(f"  Stratified (~54 neg): depth={dt_strat.get_depth()}, leaves={dt_strat.get_n_leaves()}, nodes={dt_strat.tree_.node_count}")
print(f"\nFeatures used:")
print(f"  Temporal : 11/16")
print(f"  Stratified: {(dt_strat.feature_importances_ > 0).sum()}/16")
print("\nNOTE: angkatan dan program di-drop dari fitur di stratified model.")